# Kronos-TH — Notebook 01: Data Layer Verification

**Goal:** verify that the free data sources can actually deliver clean daily OHLCV for the full Thai-retail investable universe.

**Don't trust me, run it.** Every cell here is a check.

---
## What we're checking

1. **yfinance reachability** — can Colab actually pull data?
2. **Thai stocks** (`.BK`) — history depth, gaps, volume sanity
3. **US stocks & ETFs** — same checks
4. **Crypto** — 24/7, no business-day gaps
5. **Commodities & FX** — gold, oil, USDTHB
6. **Kronos-format output** — schema matches what `KronosPredictor.predict()` expects
7. **Cache to disk** — parquet round-trip

**Honest caveats up front:**
- yfinance is free, but Yahoo throttles aggressively. We use modest delays.
- For Thai stocks, history before ~2000 is patchy. We'll use `period='10y'` as a sane default.
- Crypto trades 24/7; stocks don't. We do NOT align calendars across asset classes — each ticker keeps its own trading days.
- Some tickers in the universe may fail (delisted, ticker changed). That's fine; we log and continue.

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q yfinance pyarrow

# Mount the kth package. On Colab you'd typically clone your repo or upload
# the kth/ folder; for this notebook we assume it's in the working dir.
import sys
import os

# If you uploaded the kth folder next to this notebook:
if os.path.isdir("./kth"):
    sys.path.insert(0, ".")
    print("Found ./kth")
else:
    raise RuntimeError("Upload the kth/ folder (with data/universe.py and data/loader.py) before running.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np

from kth.data.universe import UNIVERSE, get_all_tickers_including_features, get_ticker_class, FRICTION
from kth.data.loader import to_kronos_format, quality_report, download_universe, load_cached, list_cached

print(f"yfinance: {yf.__version__}")
print(f"pandas:   {pd.__version__}")
print(f"Universe: {len(get_all_tickers_including_features())} tickers across {len(UNIVERSE)} asset classes")

## 1. Reachability smoke test

Before anything else, prove Yahoo Finance is reachable. If this cell fails, **stop** — you'll need to switch network or use a different runtime.

In [ ]:
test = yf.download("AAPL", period="5d", interval="1d", progress=False, auto_adjust=True)
if test is None or test.empty:
    raise RuntimeError("yfinance returned nothing for AAPL. Network blocked or rate-limited.")
print(f"AAPL last 5d OK — latest close: {test['Close'].iloc[-1].item():.2f}")
test

## 2. Probe each asset class

Pull just a few representatives from each class. This is the fast check before downloading the whole universe.

In [ ]:
probes = {
    "thai_equity":  ["PTT.BK", "KBANK.BK", "DELTA.BK"],
    "thai_index":   ["^SET.BK"],
    "us_equity":    ["AAPL", "NVDA"],
    "etf_global":   ["SPY", "QQQ", "VWO"],
    "commodity":    ["GLD", "GC=F"],
    "crypto":       ["BTC-USD", "ETH-USD"],
    "bond_proxy":   ["TLT"],
    "reit":         ["VNQ", "CPNREIT.BK"],
    "fx_macro":     ["THB=X"],
}

rows = []
for cls, tickers in probes.items():
    for t in tickers:
        try:
            df = yf.download(t, period="5y", interval="1d",
                             progress=False, auto_adjust=True, threads=False)
            if df is None or df.empty:
                rows.append({"class": cls, "ticker": t, "rows": 0, "start": None, "end": None})
                continue
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            rows.append({
                "class":  cls,
                "ticker": t,
                "rows":   len(df),
                "start":  df.index.min().strftime("%Y-%m-%d"),
                "end":    df.index.max().strftime("%Y-%m-%d"),
                "nan_pct": round(df.isna().mean().mean() * 100, 2),
            })
        except Exception as e:
            rows.append({"class": cls, "ticker": t, "rows": 0, "error": str(e)[:60]})

pd.DataFrame(rows)

**What to look for:**
- Every ticker should have ~1250 rows for 5y of business days (or ~1825 for crypto, which trades 7 days/wk).
- `nan_pct` should be near 0.
- If a Thai ticker has way fewer rows, it may have IPO'd recently — that's fine, just note it.
- If `^SET.BK` (the index itself) fails, try alternative symbols: `^SETI`, `SET.BK`.

## 3. Visual sanity check — plot one from each class

In [ ]:
import matplotlib.pyplot as plt

show = ["PTT.BK", "AAPL", "SPY", "GLD", "BTC-USD", "THB=X"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, t in zip(axes.flat, show):
    df = yf.download(t, period="5y", progress=False, auto_adjust=True, threads=False)
    if df is None or df.empty:
        ax.set_title(f"{t}: no data")
        continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    ax.plot(df.index, df["Close"], linewidth=1)
    ax.set_title(f"{t} — 5y close")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Download the full universe to cache

Use `period='10y'` for proper training depth. This takes a few minutes (51 tickers, with pauses to avoid throttling).

On Colab the cache lives in the runtime; you'll want to copy to Google Drive afterwards (see next cell).

In [ ]:
CACHE_DIR = "./data/raw"  # set to a Drive path for persistence

report = download_universe(
    get_all_tickers_including_features(),
    period="10y",
    cache_dir=CACHE_DIR,
    pause_between=0.5,
)
report

### Optional: persist cache to Google Drive

Colab runtimes evaporate. If you want the data to survive, mount Drive:

In [ ]:
# Uncomment to persist:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/kronos-th/data/raw
# !cp -r ./data/raw/* /content/drive/MyDrive/kronos-th/data/raw/
# print('Saved to Drive')

## 5. Quality report

In [ ]:
report['asset_class'] = report['ticker'].apply(get_ticker_class)

print("=== Per-class summary ===")
ok = report[report['rows'] > 0].copy()
summary = ok.groupby('asset_class').agg(
    n=('ticker','count'),
    mean_rows=('rows','mean'),
    min_rows=('rows','min'),
    zero_vol_days=('zero_vol_days','sum') if 'zero_vol_days' in ok.columns else ('rows','count'),
).round(0).astype({'n':int, 'mean_rows':int, 'min_rows':int})
summary

In [ ]:
# Any failures?
failed = report[report['rows'] == 0]
if len(failed) == 0:
    print(f"All {len(report)} tickers downloaded successfully.")
else:
    print(f"Failed tickers: {len(failed)}")
    failed

## 6. Load one and confirm Kronos-ready schema

In [ ]:
sample = load_cached("PTT.BK", cache_dir=CACHE_DIR)
print(f"Shape: {sample.shape}")
print(f"Cols:  {list(sample.columns)}")
print(f"Date range: {sample['timestamps'].min().date()} — {sample['timestamps'].max().date()}")
sample.tail()

In [ ]:
# Build inputs in exactly the form KronosPredictor.predict() wants
lookback, pred_len = 400, 20

x_df = sample.iloc[-(lookback+pred_len):-pred_len][
    ["open","high","low","close","volume","amount"]
].reset_index(drop=True)
x_ts = sample.iloc[-(lookback+pred_len):-pred_len]["timestamps"].reset_index(drop=True)
y_ts = sample.iloc[-pred_len:]["timestamps"].reset_index(drop=True)

print(f"x_df:        {x_df.shape}  cols={list(x_df.columns)}")
print(f"x_timestamp: {x_ts.shape}   last={x_ts.iloc[-1].date()}")
print(f"y_timestamp: {y_ts.shape}   {y_ts.iloc[0].date()}..{y_ts.iloc[-1].date()}")

---
## What's next

If everything above ran clean, the data layer is **done**. Next notebooks (to be built):

- **02_kronos_zero_shot.ipynb** — load `Kronos-small` from Hugging Face, run forecasts on a handful of tickers across asset classes, sanity-check what Kronos thinks of crypto vs Thai stocks out-of-the-box.
- **03_walkforward_backtest.ipynb** — a proper walk-forward eval with Thai-retail-realistic costs (FRICTION dict).
- **04_finetune_colab.ipynb** — fine-tune `Kronos-small` on the full universe (fits on a T4 with batch size 8 + grad accumulation).
- **05_decision_report.ipynb** — generate a daily report: for each ticker, the median forecast, the spread of probabilistic samples, and a simple signal.

Confirm Notebook 01 works on real data, then we move on.